In [2]:
import numpy as np
import pandas as pd
from pathlib import Path


# ============================================================
# 1. Peer alpha construction
# ============================================================

def add_peer_alphas(df):
    """
    Create peer's alpha features row-by-row.

    Exported peer alphas:
        peer_VWOF
        peer_MicroPrice
        peer_MicroMomentum
        peer_Spread
        peer_Spread_Limit
        peer_spread_safe
        peer_event_frac_in_min
    """

    df = df.copy()

    # Make sure time columns exist
    if "Time_dt" not in df.columns:
        df["Time_dt"] = pd.to_datetime(df["Time"], format="%H:%M:%S.%f", errors="coerce")

    df = df.dropna(subset=["Time_dt"]).copy()
    df = df.sort_values(["stock", "Time_dt"]).reset_index(drop=True)

    df["Minute"] = df["Time_dt"].dt.floor("min")

    # ------------------------------------------------------------
    # A. VWOF: volume-weighted order flow
    # ------------------------------------------------------------

    direction = df["Direction_1=Buy_-1=Sell"].astype(float)
    size = df["Size"].astype(float)

    is_new = df["NewLimitOrder_1=Yes_0=No"].astype(int)
    is_cancel = (
        df["PartialCancel_1=Yes_0=No"].astype(int)
        | df["FullDelete_1=Yes_0=No"].astype(int)
    )

    is_exec = (
        df["VisibleExecution_1=Yes_0=No"].astype(int)
        | df["HiddenExecution_1=Yes_0=No"].astype(int)
    )

    flow = np.zeros(len(df))

    flow = np.where(is_new == 1, direction * size, flow)
    flow = np.where(is_cancel == 1, -direction * size, flow)
    flow = np.where(is_exec == 1, -direction * size, flow)

    df["peer_flow"] = flow
    df["peer_abs_flow"] = np.abs(flow)

    df["peer_net_flow_50"] = (
        df.groupby("stock")["peer_flow"]
        .rolling(50, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
    )

    df["peer_total_vol_50"] = (
        df.groupby("stock")["peer_abs_flow"]
        .rolling(50, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
    )

    df["peer_VWOF"] = np.divide(
        df["peer_net_flow_50"],
        df["peer_total_vol_50"],
        out=np.zeros(len(df)),
        where=df["peer_total_vol_50"].to_numpy() != 0
    )

    # ------------------------------------------------------------
    # B. MicroMomentum
    # ------------------------------------------------------------

    df["peer_MicroPrice"] = (
        df["BidPrice_1"] * df["AskSize_1"]
        + df["AskPrice_1"] * df["BidSize_1"]
    ) / (
        df["BidSize_1"] + df["AskSize_1"]
    )

    df["peer_MidPrice"] = (
        df["BidPrice_1"] + df["AskPrice_1"]
    ) / 2

    # Same logic as peer code: shift by 1
    # But do it within each stock to avoid cross-stock leakage
    df["peer_MicroMomentum"] = (
        df["peer_MicroPrice"] - df["peer_MidPrice"]
    )

    df["peer_MicroMomentum"] = (
        df.groupby("stock")["peer_MicroMomentum"]
        .shift(1)
    )

    # ------------------------------------------------------------
    # C. Spread and spread limit
    # ------------------------------------------------------------

    df["peer_Spread"] = df["AskPrice_1"] - df["BidPrice_1"]

    df["peer_Spread_Limit"] = (
        df.groupby("stock")["peer_Spread"]
        .rolling(300, min_periods=1)
        .quantile(0.70)
        .reset_index(level=0, drop=True)
    )

    df["peer_spread_safe"] = (
        df["peer_Spread"] <= df["peer_Spread_Limit"]
    ).astype(int)

    # ------------------------------------------------------------
    # D. Time progress inside each stock-minute
    # ------------------------------------------------------------

    df["peer_event_idx_in_min"] = df.groupby(["stock", "Minute"]).cumcount()
    df["peer_n_events_in_min"] = df.groupby(["stock", "Minute"])["Time"].transform("size")

    df["peer_event_frac_in_min"] = (
        df["peer_event_idx_in_min"]
        / np.maximum(df["peer_n_events_in_min"] - 1, 1)
    )

    # Clean missing values
    peer_alpha_cols = [
        "peer_VWOF",
        "peer_MicroPrice",
        "peer_MicroMomentum",
        "peer_Spread",
        "peer_Spread_Limit",
        "peer_spread_safe",
        "peer_event_frac_in_min",
    ]

    df[peer_alpha_cols] = (
        df[peer_alpha_cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    return df, peer_alpha_cols

In [3]:
# ============================================================
# 2. Load original orderbook files and export peer alpha data
# ============================================================

data_path = Path("../Project_Train_Datasets")

file_map = {
    "AMZN": data_path / "AMZN_5levels_train.csv",
    "GOOG": data_path / "GOOG_5levels_train.csv",
    "INTC": data_path / "INTC_5levels_train.csv",
    "MSFT": data_path / "MSFT_5levels_train.csv",
}


def load_orderbook_for_peer_alphas(file_map):
    dfs = []

    for stock, path in file_map.items():
        temp = pd.read_csv(path)
        temp["stock"] = stock
        dfs.append(temp)

    df = pd.concat(dfs, ignore_index=True)

    return df


raw_df = load_orderbook_for_peer_alphas(file_map)

peer_df, peer_alpha_cols = add_peer_alphas(raw_df)

export_cols = [
    "stock",
    "Time",
    "OrderID",
] + peer_alpha_cols

peer_alpha_export = peer_df[export_cols].copy()

output_path = Path("peer_alpha_data.csv")
peer_alpha_export.to_csv(output_path, index=False)

print("Saved peer alpha data to:", output_path.resolve())
print("Shape:", peer_alpha_export.shape)
print("Peer alpha columns:")
print(peer_alpha_cols)
print(peer_alpha_export.head())

Saved peer alpha data to: /Users/chenyu/Desktop/ORIE5259 Market Microstructure/ORIE-5259-Algo-Trading/Strategy_5/peer_alpha_data.csv
Shape: (1149529, 10)
Peer alpha columns:
['peer_VWOF', 'peer_MicroPrice', 'peer_MicroMomentum', 'peer_Spread', 'peer_Spread_Limit', 'peer_spread_safe', 'peer_event_frac_in_min']
  stock          Time   OrderID  peer_VWOF  peer_MicroPrice  \
0  AMZN  09:30:00.017         0   1.000000       223.565000   
1  AMZN  09:30:00.189  11885113   1.000000       223.834298   
2  AMZN  09:30:00.189   3911376   0.047619       223.834298   
3  AMZN  09:30:00.189  11534792   0.718310       223.834298   
4  AMZN  09:30:00.189   1365373   0.574194       223.834298   

   peer_MicroMomentum  peer_Spread  peer_Spread_Limit  peer_spread_safe  \
0            0.000000         0.77              0.770                 1   
1            0.000000         0.14              0.581                 1   
2           -0.045702         0.14              0.392                 1   
3         